# Claude Code in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexbui96/alexbui96/blob/claude/claude-google-colab-60npvy/colab/claude_code_in_colab.ipynb)

Run the **Claude Code CLI** and the **Claude Agent SDK** inside a Colab runtime, so Claude can read,
write and execute files on the Colab VM — refactor a script, fix a failing test, build a small project —
instead of just returning text into a chat cell.

**What you need**

- An Anthropic **Console API key** (`sk-ant-...`) from <https://platform.claude.com>, stored in Colab Secrets.
  Colab has no browser-based OAuth login, so a Console key is the practical path here.
- A Claude **Pro, Max, Team, Enterprise or Console** account. The free claude.ai plan does not include Claude Code.
- No GPU. Claude Code does no local inference — a plain CPU runtime is right, and cheaper on your Colab quota.

**What this notebook covers**

| Section | What you get |
| --- | --- |
| 1–3 | Install the CLI, wire up the API key, verify |
| 4–5 | First headless run; letting Claude edit real files |
| 6–8 | JSON output, cost tracking, multi-turn sessions, live streaming |
| 9 | A reusable `claude_run()` Python helper |
| 10 | The Claude Agent SDK (`query`, `ClaudeSDKClient`) with Colab's event loop |
| 11 | Gotchas worth knowing before you burn a run |

## 1. Install the Claude Code CLI

Use the native installer. It drops a self-contained binary at `~/.local/bin/claude` and needs **no Node.js**,
which keeps it independent of whatever Colab's preinstalled Node happens to be.

(The npm route, `npm install -g @anthropic-ai/claude-code`, also works but wants Node.js 22+.)

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash

The installer writes to `~/.local/bin`, which is not on Colab's `PATH` by default.
Setting it in `os.environ` is enough: every later `!` cell and `subprocess` call inherits it.

Re-run this cell (not the installer) if you ever restart the runtime *without* recycling the VM.

In [ ]:
import os

local_bin = os.path.expanduser('~/.local/bin')
if local_bin not in os.environ['PATH'].split(':'):
    os.environ['PATH'] = f"{local_bin}:{os.environ['PATH']}"

!which claude && claude --version

## 2. Give it your API key

Put the key in **Colab Secrets** (🔑 in the left sidebar) under the name `ANTHROPIC_API_KEY`,
then toggle *Notebook access* on. That keeps the key out of the notebook file, out of your outputs,
and out of git.

The cell falls back to a hidden prompt if you are running somewhere other than Colab.

> Never paste the key into a cell as a literal. Notebook outputs and `.ipynb` files get committed, shared and indexed.

In [ ]:
import getpass
import os

if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
        print('Key loaded from Colab Secrets.')
    except Exception as exc:
        print(f'Colab Secrets unavailable ({type(exc).__name__}); falling back to a prompt.')
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

key = os.environ['ANTHROPIC_API_KEY']
assert key.startswith('sk-ant-'), 'That does not look like an Anthropic API key.'
print(f'Key ending in ...{key[-4:]} is set.')

## 3. Verify the install

`claude doctor` prints read-only diagnostics — install health, settings problems, auto-update state —
without starting a session or spending a token.

In [ ]:
!claude doctor

## 4. Your first headless run

`-p` (print) is non-interactive mode: one prompt in, the answer on stdout, then exit.

Two flags matter from the start:

- **`--bare`** skips auto-discovery of hooks, skills, plugins, MCP servers and `CLAUDE.md`. It starts faster and,
  more importantly, makes the run reproducible — nothing in the working directory can quietly change its behavior.
  It is the recommended mode for scripted calls and reads `ANTHROPIC_API_KEY` directly.
- **`--allowedTools`** pre-approves tools. A `-p` session starts in Manual permission mode, so anything you
  don't pre-approve is simply denied — nobody is there to click *Allow*.

In [ ]:
!echo '# Scratch project' > /content/README.md
!claude --bare -p 'Read README.md and describe this project in one sentence.' \
    --allowedTools 'Read' \
    --add-dir /content

## 5. Let Claude actually work on files

This is the part a plain API call can't do: Claude reads, writes and runs code in the runtime.

**Permission modes** decide how much it can do unattended:

| Mode | Behavior |
| --- | --- |
| `manual` | The `-p` default. Everything that would prompt gets denied. |
| `acceptEdits` | Writes files without asking, plus common filesystem commands (`mkdir`, `mv`, `cp`). Other shell commands still need an `--allowedTools` entry. |
| `dontAsk` | Denies every call that would prompt; good for locked-down runs. |
| `auto` | A classifier reviews each action instead of you. |
| `plan` | Claude plans but changes nothing. |
| `bypassPermissions` | No checks at all (`--dangerously-skip-permissions`). |

`acceptEdits` plus an explicit `--allowedTools` list is the right default for a notebook: Claude can build
and run what you asked for, and the tool list is a short, readable statement of what it may reach for.
Reach for `bypassPermissions` only in a runtime you are happy to throw away — and note Colab runs as root,
where that flag may refuse to start.

Pair it with `--permission-prompts none` so nothing can sit waiting for an answer that will never come,
and `--max-turns` so a confused run ends instead of looping.

In [ ]:
%%bash
mkdir -p /content/demo && cd /content/demo

claude --bare -p 'Write fizzbuzz.py with a fizzbuzz(n) function returning a list of strings, 
and test_fizzbuzz.py with pytest tests for it. Then run pytest and fix anything that fails.' \
    --permission-mode acceptEdits \
    --permission-prompts none \
    --allowedTools 'Read,Write,Edit,Bash(python *),Bash(pytest *),Bash(pip install *)' \
    --max-turns 15 \
    --model claude-sonnet-5

In [ ]:
!ls -la /content/demo && echo '--- fizzbuzz.py ---' && cat /content/demo/fizzbuzz.py

### Picking a model

`--model` takes an alias (`opus`, `sonnet`, `haiku`, `fable`) or a full ID:

| Model | ID | Good for |
| --- | --- | --- |
| Claude Opus 5 | `claude-opus-5` | Hard, multi-step agentic work |
| Claude Sonnet 5 | `claude-sonnet-5` | The everyday default — fast, cheap enough to iterate |
| Claude Haiku 4.5 | `claude-haiku-4-5` | Bulk, simple, latency-sensitive runs |

Model IDs are complete as written — never append a date suffix.

## 6. Structured output and cost

`--output-format json` returns the answer plus session metadata, which is what makes the CLI usable
*from* Python rather than just alongside it. The payload includes `result`, `session_id`, `num_turns`,
`duration_ms` and `total_cost_usd`.

`total_cost_usd` is a client-side estimate, so treat it as a running gauge, not an invoice.

In [ ]:
import json
import subprocess

proc = subprocess.run(
    ['claude', '--bare', '-p', 'In one sentence: what does fizzbuzz.py do?',
     '--output-format', 'json',
     '--allowedTools', 'Read',
     '--model', 'claude-sonnet-5'],
    cwd='/content/demo', capture_output=True, text=True,
)

payload = json.loads(proc.stdout)
print(payload['result'])
print()
print(f"session : {payload['session_id']}")
print(f"turns   : {payload['num_turns']}")
print(f"seconds : {payload['duration_ms'] / 1000:.1f}")
print(f"cost    : ${payload['total_cost_usd']:.4f} (estimate)")

### Force a JSON shape

`--json-schema` constrains the answer to a schema you define. The parsed object lands in
`structured_output`, so you can feed it straight into pandas or the rest of your pipeline.

In [ ]:
import json
import subprocess

schema = {
    'type': 'object',
    'properties': {
        'functions': {'type': 'array', 'items': {'type': 'string'}},
        'has_tests': {'type': 'boolean'},
    },
    'required': ['functions', 'has_tests'],
}

proc = subprocess.run(
    ['claude', '--bare', '-p', 'List the function names defined in this project.',
     '--output-format', 'json',
     '--json-schema', json.dumps(schema),
     '--allowedTools', 'Read,Glob,Grep',
     '--model', 'claude-sonnet-5'],
    cwd='/content/demo', capture_output=True, text=True,
)

print(json.loads(proc.stdout)['structured_output'])

## 7. Multi-turn: keep the same session

Each `-p` call is its own session unless you say otherwise. Capture `session_id` and pass it to `--resume`
to continue with the full conversation context — Claude remembers the files it already read, so follow-ups
are cheaper and better grounded.

`--continue` picks up the most recent conversation in the current directory if you'd rather not track IDs.

In [ ]:
import json
import subprocess


def ask(prompt, session_id=None, cwd='/content/demo'):
    cmd = ['claude', '--bare', '-p', prompt,
           '--output-format', 'json',
           '--allowedTools', 'Read,Glob,Grep',
           '--model', 'claude-sonnet-5']
    if session_id:
        cmd += ['--resume', session_id]
    payload = json.loads(subprocess.run(cmd, cwd=cwd, capture_output=True, text=True).stdout)
    return payload['result'], payload['session_id']


answer, session = ask('What edge cases does the test file cover?')
print(answer, '\n')

answer, session = ask('Which edge case is missing?', session_id=session)
print(answer)

## 8. Stream the run as it happens

A long agentic run is otherwise a blank cell for several minutes. `--output-format stream-json`
emits newline-delimited events; print them as they arrive and you get a live trace of Claude's tool calls.

`--verbose` is required with `stream-json`, and `--include-partial-messages` adds token-level deltas.

In [ ]:
import json
import subprocess

cmd = ['claude', '--bare', '-p',
       'Add a docstring to fizzbuzz() explaining the rules, then re-run the tests.',
       '--output-format', 'stream-json', '--verbose',
       '--permission-mode', 'acceptEdits',
       '--permission-prompts', 'none',
       '--allowedTools', 'Read,Edit,Bash(pytest *)',
       '--max-turns', '10',
       '--model', 'claude-sonnet-5']

with subprocess.Popen(cmd, cwd='/content/demo', stdout=subprocess.PIPE,
                      text=True, bufsize=1) as proc:
    for line in proc.stdout:
        line = line.strip()
        if not line:
            continue
        try:
            event = json.loads(line)
        except json.JSONDecodeError:
            continue

        kind = event.get('type')
        if kind == 'system' and event.get('subtype') == 'init':
            print(f"[init] model={event.get('model')}")
        elif kind == 'assistant':
            for block in event['message']['content']:
                if block['type'] == 'text' and block['text'].strip():
                    print(f"[claude] {block['text'].strip()}")
                elif block['type'] == 'tool_use':
                    print(f"[tool] {block['name']}")
        elif kind == 'result':
            print(f"\n[done] ${event.get('total_cost_usd', 0):.4f} "
                  f"in {event.get('num_turns')} turns")
            print(event.get('result', ''))

## 9. A reusable helper

Everything above, wrapped once. Drop this into any Colab notebook where you want Claude working on files.

In [ ]:
import json
import subprocess
from typing import Optional


def claude_run(
    prompt: str,
    cwd: str = '/content',
    tools: str = 'Read,Glob,Grep',
    model: str = 'claude-sonnet-5',
    edit: bool = False,
    resume: Optional[str] = None,
    max_turns: int = 20,
    timeout: int = 900,
) -> dict:
    """Run Claude Code headlessly and return the parsed JSON payload.

    Set edit=True to let Claude write files (acceptEdits mode). Returns a dict with
    'result', 'session_id', 'num_turns' and 'total_cost_usd'.
    """
    cmd = ['claude', '--bare', '-p', prompt,
           '--output-format', 'json',
           '--allowedTools', tools,
           '--model', model,
           '--max-turns', str(max_turns),
           '--permission-prompts', 'none']
    if edit:
        cmd += ['--permission-mode', 'acceptEdits']
    if resume:
        cmd += ['--resume', resume]

    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr or proc.stdout)
    return json.loads(proc.stdout)


out = claude_run('Summarize every file in this directory in one line each.', cwd='/content/demo')
print(out['result'])

## 10. The Claude Agent SDK

The Python SDK is the same harness as the CLI, exposed as objects instead of flags: typed messages,
tool-approval callbacks, and a session you can hold open. Worth it when the notebook is doing more
than shelling out.

It drives the CLI under the hood, so **keep the install from section 1** — the SDK alone is not enough.

In [ ]:
!pip install -q claude-agent-sdk

### The one Colab-specific trap

Colab already runs an event loop, so **`asyncio.run(...)` raises** `RuntimeError: asyncio.run() cannot be
called from a running event loop`. Every SDK example you'll find online ends with `asyncio.run(main())`.

In a notebook, `await main()` directly instead — Colab supports top-level `await`.

In [ ]:
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    TextBlock,
    query,
)

options = ClaudeAgentOptions(
    model='claude-sonnet-5',
    permission_mode='acceptEdits',
    allowed_tools=['Read', 'Write', 'Edit', 'Bash'],
    cwd='/content/demo',
    max_turns=10,
)


async def run(prompt):
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
        elif isinstance(message, ResultMessage):
            print(f'\n[cost] ${message.total_cost_usd:.4f}')


# Top-level await — do NOT wrap this in asyncio.run() inside Colab.
await run('Add a __main__ block to fizzbuzz.py that prints fizzbuzz(20).')

### Holding a session open

`ClaudeSDKClient` keeps one session across several exchanges, so context carries over without you
threading session IDs around. It also supports interrupts, which `query()` does not.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient, ResultMessage


async def conversation():
    async with ClaudeSDKClient(options=options) as client:
        await client.query('What does this project do?')
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print('Q1:', message.result, '\n')

        await client.query('What would you add to it next?')
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print('Q2:', message.result)


await conversation()

## 11. Gotchas

**The runtime is disposable.** Colab reclaims the VM after idle time or ~12 hours, and everything under
`/content` goes with it. Push to git or mount Drive before you disconnect:

```python
from google.colab import drive
drive.mount('/content/drive')
```

**You are root.** Colab has no unprivileged user, which is why `acceptEdits` plus an explicit `--allowedTools`
list beats reaching for `--dangerously-skip-permissions`. There is no second line of defense in this VM.

**Reinstall after a factory reset.** *Runtime → Disconnect and delete runtime* wipes the binary. Re-run
section 1. A plain *Restart runtime* only needs the `PATH` cell.

**Every run costs money.** Colab's free tier does not subsidize Anthropic API calls. Watch `total_cost_usd`,
keep `--max-turns` low while iterating, and prefer `claude-sonnet-5` or `claude-haiku-4-5` for routine work.

**Interactive mode is unusable here.** Bare `claude` wants a TTY; Colab cells don't provide one. `-p` is the
whole interface. For an interactive terminal, use Colab's own terminal (paid tiers) or run Claude Code locally.

**Secrets stay in Secrets.** Clear outputs before committing a notebook — a key echoed once into a cell
output lives in the `.ipynb` forever.

**Long runs versus tab sleep.** A closed or backgrounded Colab tab can disconnect mid-run and you lose the
output. Keep big agentic tasks under a few minutes, or stream (section 8) so partial progress is visible.

---

### Where to go next

- [Claude Code docs](https://code.claude.com/docs/en/overview)
- [CLI reference](https://code.claude.com/docs/en/cli-reference) — every flag
- [Run Claude Code programmatically](https://code.claude.com/docs/en/headless)
- [Agent SDK (Python)](https://code.claude.com/docs/en/agent-sdk/python)